# 2. First Round Bayesian Model

This notebook builds, samples, and diagnoses the first-round Dirichlet-Multinomial model. It covers:
- Model graph: random walk, house effects, election likelihood
- MCMC sampling and convergence diagnostics
- Posterior interpretation

**References**: `docs/architecture/model-design.md` | `MVP_SPECS_GUIDE.md` §9

In [ ]:
import os, sys
# Ensure CWD is the project root (parent of notebooks/)
if os.path.basename(os.getcwd()) in ("notebooks", ""):
    os.chdir("..")
sys.path.insert(0, ".")


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, arviz as az, matplotlib.pyplot as plt

from co_president.config import ModelConfig, FIRST_ROUND_CANDIDATES, ELECTION_DATE_ROUND1
from co_president.data import load_and_clean_all, load_canonical_results
from co_president.model_round1 import build_round1_model, sample_round1, forecast_round1

plt.rcParams['figure.dpi'] = 100
CONFIG = ModelConfig(mcmc_draws=500, mcmc_tune=300, mcmc_chains=2, mcmc_cores=2, target_accept=0.95, seed=332211, nuts_sampler='numpyro')
cp = load_and_clean_all(); r1, r2 = load_canonical_results()
ds = pd.read_parquet('results/trends_cache_2022.parquet'); ds['fecha'] = pd.to_datetime(ds['fecha'])


## Build the Model

The model has these components:
- **Reverse-time random walk**: vote intentions evolve backward from election day
- **House effects**: each pollster gets a per-candidate bias (zero-sum constrained)
- **Election likelihood**: fixed-concentration DirichletMultinomial (φ=5000)
- **Digital signal likelihood**: Beta observations from Google Trends

In [ ]:
model = build_round1_model(cp.round1, r1, CONFIG, digital_signals=ds)
print("Free RVs:", len(model.free_RVs))
for rv in model.free_RVs:
    print(f"  {rv.name}: shape {rv.shape}")


## Sample with numpyro

In [ ]:
idata = sample_round1(model, CONFIG)


## Convergence Diagnostics

Check R-hat (should be < 1.05) and ESS (should be > 100).

In [ ]:
rhat = az.rhat(idata.posterior)
rhat_max = max(float(v.max()) for v in rhat.values())
ess_bulk = az.ess(idata.posterior, method='bulk')
ess_min = min(float(v.min()) for v in ess_bulk.values())
div = idata.sample_stats.diverging.sum().values
print(f"Max R-hat:  {rhat_max:.4f}")
print(f"Min ESS:    {ess_min:.0f}")
print(f"Divergences: {int(div)}")


## Trace Plot

In [ ]:
az.plot_trace(idata, var_names=["sigma_rw", "sigma_house", "phi_poll"])
import matplotlib.pyplot as plt
plt.tight_layout()


## Posterior Predictions

In [ ]:
candidate_keys = sorted(set(FIRST_ROUND_CANDIDATES.keys()) & set(cp.round1.columns))
fc = forecast_round1(idata, candidate_keys)
print(f"{'Candidate':25s} {'Predicted':>9s} {'Actual':>9s} {'Error':>9s} {'95% CI':>20s}")
print('-' * 72)
for cf in fc.candidates:
    err = cf.mean_share - r1.get_share(cf.candidate_key)
    ci = cf.ci_95
    print(f"{cf.candidate_key:25s} {cf.mean_share*100:7.2f}% {r1.get_share(cf.candidate_key)*100:7.2f}% "
          f"{err*100:+7.2f}pp [{ci[0]*100:5.1f}%, {ci[1]*100:5.1f}%]")


## House Effects

Which pollsters over/under-estimate which candidates?

In [ ]:
house = idata.posterior['house_effects']
house_mean = house.mean(dim=('chain','draw')).to_numpy()
pollsters = model.coords.get('pollster_dim', [f'P{i}' for i in range(house_mean.shape[0])])
fig, ax = plt.subplots(figsize=(8, max(4, len(pollsters)*0.3)))
im = ax.imshow(house_mean, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
ax.set_yticks(range(len(pollsters))); ax.set_yticklabels(pollsters)
ax.set_xticks(range(len(candidate_keys))); ax.set_xticklabels([c[:8] for c in candidate_keys], rotation=45)
ax.set_title('House Effects (logit scale)')
plt.colorbar(im)
plt.tight_layout()
